
# 백화점 가상 데이터 분석 노트북

이 노트북은 `fake_department_store_data.csv` 파일을 불러와서 기본 점검, 전처리, 핵심 집계, 시각화를 한 번에 볼 수 있도록 정리한 예시입니다.

## 포함 내용
- 데이터 불러오기
- 컬럼/결측치/기초통계 확인
- 날짜형 변환
- 카테고리별/브랜드별 매출 분석
- 객단가(`sales_per_customer`) 확인
- 간단한 시각화

> VS Code에서 `.ipynb` 파일을 열면 셀 단위로 바로 실행할 수 있습니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)

file_path = 'fake_department_store_data.csv'
df = pd.read_csv(file_path)

print(f'행 수: {df.shape[0]:,}, 열 수: {df.shape[1]}')
df.head()


## 1. 데이터 구조 확인

먼저 컬럼명, 타입, 결측치, 기초 통계를 확인합니다.


In [ ]:
print('컬럼 목록')
display(pd.DataFrame({'컬럼명': df.columns, '데이터타입': df.dtypes.astype(str).values}))

print('\n기본 정보')
df.info()

print('\n결측치 개수')
display(df.isna().sum().to_frame('결측치 수'))

print('\n기초 통계')
display(df.describe(include='all').T)


## 2. 전처리

`date` 컬럼을 날짜형으로 변환하고, 분석에 쓰기 편하도록 파생 컬럼을 추가합니다.


In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['year_month'] = df['date'].dt.to_period('M').astype(str)

day_map = {
    'Monday': '월요일', 'Tuesday': '화요일', 'Wednesday': '수요일',
    'Thursday': '목요일', 'Friday': '금요일', 'Saturday': '토요일', 'Sunday': '일요일'
}
df['day_of_week'] = df['date'].dt.day_name().map(day_map)

(df.head())


## 3. 전체 매출/고객/객단가 요약


In [ ]:
summary = pd.DataFrame({
    '총매출': [df['sales'].sum()],
    '총고객수': [df['customers'].sum()],
    '평균매출': [df['sales'].mean()],
    '평균고객수': [df['customers'].mean()],
    '평균객단가': [df['sales_per_customer'].mean()]
})

summary.style.format({
    '총매출': '{:,.0f}',
    '총고객수': '{:,.0f}',
    '평균매출': '{:,.1f}',
    '평균고객수': '{:,.1f}',
    '평균객단가': '{:,.2f}'
})


## 4. 카테고리별 분석

카테고리 기준으로 총매출, 고객수, 평균 객단가를 확인합니다.


In [ ]:
category_summary = (
    df.groupby('category', as_index=False)
      .agg(
          총매출=('sales', 'sum'),
          총고객수=('customers', 'sum'),
          평균매출=('sales', 'mean'),
          평균객단가=('sales_per_customer', 'mean')
      )
      .sort_values('총매출', ascending=False)
      .rename(columns={'category': '카테고리'})
)

category_summary.style.format({
    '총매출': '{:,.0f}',
    '총고객수': '{:,.0f}',
    '평균매출': '{:,.1f}',
    '평균객단가': '{:,.2f}'
})

In [ ]:
category_sales = df.groupby('category')['sales'].sum().sort_values(ascending=False)

ax = category_sales.plot(kind='bar')
ax.set_title('카테고리별 총매출')
ax.set_xlabel('카테고리')
ax.set_ylabel('매출')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. 브랜드별 분석

브랜드별 총매출과 평균 객단가를 확인합니다.


In [ ]:
brand_summary = (
    df.groupby('brand', as_index=False)
      .agg(
          총매출=('sales', 'sum'),
          총고객수=('customers', 'sum'),
          평균객단가=('sales_per_customer', 'mean'),
          평균면적=('area_sqm', 'mean')
      )
      .sort_values('총매출', ascending=False)
      .rename(columns={'brand': '브랜드'})
)

brand_summary.style.format({
    '총매출': '{:,.0f}',
    '총고객수': '{:,.0f}',
    '평균객단가': '{:,.2f}',
    '평균면적': '{:,.1f}'
})

In [ ]:
top10_brand_sales = df.groupby('brand')['sales'].sum().sort_values(ascending=False).head(10)

ax = top10_brand_sales.plot(kind='bar')
ax.set_title('브랜드별 총매출 TOP 10')
ax.set_xlabel('브랜드')
ax.set_ylabel('매출')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. 일자별 추이

날짜별 매출 추이를 확인합니다.


In [ ]:
daily_sales = df.groupby('date', as_index=False)['sales'].sum()

ax = daily_sales.plot(x='date', y='sales')
ax.set_title('일자별 총매출 추이')
ax.set_xlabel('날짜')
ax.set_ylabel('매출')
plt.tight_layout()
plt.show()


## 7. 면적 대비 성과 간단 확인

`area_sqm`과 `sales`의 관계를 산점도로 봅니다.


In [ ]:
ax = df.plot.scatter(x='area_sqm', y='sales')
ax.set_title('매장 면적과 매출 관계')
ax.set_xlabel('매장 면적 (㎡)')
ax.set_ylabel('매출')
plt.tight_layout()
plt.show()


## 8. 추가로 바로 해볼 수 있는 분석 아이디어

아래 셀은 필요 시 확장해서 사용하시면 됩니다.

- 요일별 매출 비교
- 카테고리 x 브랜드 피벗 테이블
- 객단가 상위 브랜드 추출
- 이상치 탐지


In [ ]:
# 예시 1) 요일별 평균 매출
weekday_summary = (
    df.groupby('day_of_week', as_index=False)
      .agg(평균매출=('sales', 'mean'))
      .rename(columns={'day_of_week': '요일'})
      .sort_values('평균매출', ascending=False)
)
weekday_summary

In [ ]:
# 예시 2) 카테고리 x 브랜드 피벗
pivot_table = pd.pivot_table(
    df,
    index='category',
    columns='brand',
    values='sales',
    aggfunc='sum',
    fill_value=0
).rename_axis('카테고리').rename_axis('브랜드', axis=1)
pivot_table


## 9. 메모

VS Code에서 실행이 안 될 경우 아래를 먼저 확인해 주세요.

1. Python extension 설치
2. Jupyter extension 설치
3. 우측 상단 커널 선택
4. 현재 `.ipynb` 파일과 같은 폴더에 `fake_department_store_data.csv` 파일이 있는지 확인

경로가 다르면 아래처럼 수정하면 됩니다.

```python
file_path = r'C:/Users/사용자이름/Desktop/fake_department_store_data.csv'
```
